# TP5. Synthèse : chiffrer, déchiffrer, casser

Carnet de travail. L'énoncé complet, avec les corrigés, est sur le site :
https://cours.gclab.fr/outils-fondamentaux/tp5-synthese/

Exécutez une cellule avec Maj+Entrée. **Pensez à télécharger ce carnet avant de partir** : menu File, puis Download.


## Exercice 1.1 : les deux conversions

Écrivez `lettre_vers_nombre(c)` et `nombre_vers_lettre(n)`, réciproques l'une de
l'autre sur les majuscules.

**Résultat attendu :** `lettre_vers_nombre("C")` vaut `2`, et
`nombre_vers_lettre(2)` vaut `"C"`.


In [ ]:
# votre réponse ici
def lettre_vers_nombre(c):
    return ord(c.upper()) - ord("A")

def nombre_vers_lettre(n):
    return chr(n + ord("A"))

print("C ->", lettre_vers_nombre("C"))
print("2 ->", nombre_vers_lettre(2))

## Exercice 1.2 : nettoyer un texte

Écrivez `nettoyer(texte)` qui met tout en majuscules et ne garde que les lettres de A
à Z, en supprimant espaces, ponctuation et accents.

**Résultat attendu :** `nettoyer("Été, à Paris !")` vaut `"ETEAPARIS"`.


In [ ]:
# votre réponse ici
import unicodedata

def nettoyer(texte):
    
    nfkd_form = unicodedata.normalize('NFKD', texte)
    sans_accents = "".join([c for c in nfkd_form if not unicodedata.combining(c)])

    resultat = ""
    for c in sans_accents.upper():
        if 'A' <= c <= 'Z':
            resultat += c
    return resultat

print(nettoyer("Été, à Paris !"))

## Exercice 2.1 : la fonction de chiffrement

Écrivez `chiffrer(texte, a, b)` qui applique $E(x) = (ax + b) \bmod 26$ à chaque
lettre d'un texte déjà nettoyé.

**Résultat attendu :** `chiffrer("ATTAQUEALAUBE", 5, 8)` vaut `"IZZIKECILIENC"`.


In [ ]:
# votre réponse ici
def chiffrer(texte, a, b):
    texte_propre = nettoyer(texte)
    chiffre = ""
    for c in texte_propre:
        x = lettre_vers_nombre(c)
        y = (a * x + b) % 26
        chiffre += nombre_vers_lettre(y)
    return chiffre

print(chiffrer("ATTAQUEALAUBE", 5, 8))

## Exercice 3.1 : les clés utilisables

Écrivez le code qui donne la liste des valeurs de `a` entre 1 et 25 pour lesquelles le
chiffrement est déchiffrable. Combien y en a-t-il ?

**Résultat attendu :** douze valeurs, toutes impaires et différentes de 13.


In [ ]:
# votre réponse ici
import math

# Une clé a est valide si elle est premier avec 26 (pgcd(a, 26) == 1)
cles_valides = [a for a in range(1, 26) if math.gcd(a, 26) == 1]
print("Clés a possibles :", cles_valides)
print("Nombre de clés :", len(cles_valides))

## Exercice 3.2 : l'inverse modulaire

Écrivez `inverse_modulaire(a, n)` qui renvoie l'inverse de `a` modulo `n`, ou lève une
erreur s'il n'existe pas.

**Résultat attendu :** l'inverse de 5 modulo 26 est 21, car $5 \times 21 = 105 =
4 \times 26 + 1$.


In [ ]:
# votre réponse ici
def inverse_modulaire(a, n):
    for x in range(1, n):
        if (a * x) % n == 1:
            return x
    raise ValueError(f"Pas d'inverse pour {a} modulo {n}")

print("Inverse de 5 mod 26 :", inverse_modulaire(5, 26))

## Exercice 4.1 : la fonction de déchiffrement

Écrivez `dechiffrer(texte, a, b)`, en refusant les clés invalides. Vérifiez que
déchiffrer ce qu'on a chiffré redonne le message d'origine.

**Résultat attendu :** `dechiffrer(chiffrer(m, 5, 8), 5, 8)` redonne `m` nettoyé.


In [ ]:
# votre réponse ici
def dechiffrer(texte, a, b):
    
    if math.gcd(a, 26) != 1:
        raise ValueError("Clé a invalide (non inversible modulo 26)")
    
    a_inv = inverse_modulaire(a, 26)
    clair = ""
    for c in texte:
        y = lettre_vers_nombre(c)
        x = (a_inv * (y - b)) % 26
        clair += nombre_vers_lettre(x)
    return clair


msg = "ATTAQUEALAUBE"
crypted = chiffrer(msg, 5, 8)
print("Déchiffré :", dechiffrer(crypted, 5, 8))

## Exercice 5.1 : essayer toutes les clés

Écrivez `toutes_les_clefs(secret)` qui renvoie la liste des 312 déchiffrements
possibles avec leur clé. Retrouvez le message clair à l'œil.

**Résultat attendu :** parmi les 312 propositions, une seule est du français lisible.


In [ ]:
# votre réponse ici
def toutes_les_clefs(secret):
    resultats = []
    cles_valides = [1, 3, 5, 7, 9, 11, 15, 17, 19, 21, 23, 25]
    for a in cles_valides:
        for b in range(26):
            clair = dechiffrer(secret, a, b)
            resultats.append((a, b, clair))
    return resultats

secret_test = chiffrer("BONJOUR", 5, 8)
candidates = toutes_les_clefs(secret_test)
print(f"Total de {len(candidates)} propositions générées.")
print("Exemple :", candidates[0])

## Exercice 6.1 : casser automatiquement

Comptez les occurrences de chaque lettre du message chiffré, puis testez les clés qui
envoient E sur la lettre la plus fréquente. Retrouvez le message sans intervention
humaine.

**Résultat attendu :** le message clair, trouvé parmi une poignée de candidats au
lieu de 312.


In [ ]:
# votre réponse ici
def casser_affine(secret):
    
    compteur = {}
    for c in secret:
        compteur[c] = compteur.get(c, 0) + 1
    
 
    lettre_freq_max = max(compteur, key=compteur.get)
    y_max = lettre_vers_nombre(lettre_freq_max)
    
    
    x_e = 4
    
    candidats = []
    cles_valides = [1, 3, 5, 7, 9, 11, 15, 17, 19, 21, 23, 25]
    
    for a in cles_valides:
    
        b = (y_max - a * x_e) % 26
        clair = dechiffrer(secret, a, b)
        candidats.append((a, b, clair))
        
    return candidats

print("Candidats auto-cassés :")
for a, b, texte in casser_affine(chiffrer("CRYPTOGRAPHIE", 3, 5))[:3]:
    print(f"a={a}, b={b} -> {texte}")

## Exercice 7.1 : l'indice de coïncidence

Écrivez `indice_coincidence(texte)`. Calculez-le sur `texte_long`, sur `secret_long`,
et sur une suite de lettres tirées au hasard de même longueur.

**Résultat attendu :** la même valeur pour le clair et pour le chiffré affine, environ
0,087 sur ce texte court, contre environ 0,039 pour les lettres au hasard.


In [ ]:
# votre réponse ici


## Exercice 8.1 : chiffrer par Vigenère

Écrivez `vigenere(texte, cle)`. Chiffrez `texte_long` avec la clé `"CLE"`, et calculez
l'indice de coïncidence du résultat.

**Résultat attendu :** un indice d'environ 0,055, intermédiaire entre le français et
le hasard. L'attaque par fréquences de l'étape 6 ne s'applique plus telle quelle.


In [ ]:
# votre réponse ici


## Exercice 8.2 : retrouver la longueur de la clé

Si la clé a $L$ lettres, alors les lettres du chiffré de rangs $0, L, 2L, \dots$
ont toutes subi le même décalage : c'est un chiffre de César, et son indice de
coïncidence est celui du français. Pour chaque longueur candidate `L` de 1 à 7,
découpez le chiffré en `L` colonnes, calculez l'indice moyen des colonnes, et repérez
les longueurs où il remonte.

**Résultat attendu :** l'indice moyen bondit à environ 0,09 pour `L = 3` et
`L = 6`, et reste vers 0,055 pour les autres longueurs. La clé a donc trois lettres,
ou un multiple de trois.


In [ ]:
# votre réponse ici
